In [ ]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments
)
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, concatenate_datasets
import pandas as pd

def load_validated_samples(validation_file_path):
    corrected_df = pd.read_excel(validation_file_path)
    corrected_dataset = Dataset.from_pandas(corrected_df)
    return corrected_dataset

def load_original_data(original_data_path):
    original_df = pd.read_excel(original_data_path)
    original_dataset = Dataset.from_pandas(original_df)
    return original_dataset

def main():
    model_name = "Bartenderr/Bert_snomed"
    output_dir = "new_model_version"
    original_data_path = 'tariff_df'
    validated_data_path = 'validated_df'
    
    
    # load my model
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name)
    
    # 
    original_data = load_original_data(original_data_path)
    corrected_data = load_validated_samples(validated_data_path)
    
    # Combine datasets
    retrain_dataset = concatenate_datasets([original_data, corrected_data])
    
    # Tokenize the combined dataset
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)
    
    tokenized_dataset = retrain_dataset.map(tokenize_function, batched=True)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=3,               # Adjust as needed
        per_device_train_batch_size=8,    # Adjust based on your GPU
        save_steps=10_000,
        save_total_limit=2,
        logging_dir='./logs',
        logging_steps=500,
        evaluation_strategy="steps",      # Add if you have validation data
        eval_steps=500,                   # Add if you have validation data
        load_best_model_at_end=True,      # Add if you have validation data
    )
    
    # Initialize Trainer
    retrainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
    )
    
    # Train and save
    retrainer.train()
    retrainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"Retraining complete. New model saved to {output_dir}")

if __name__ == "__main__":
    main()